# 六大设计原则：SOLID + LoD

> 由浅入深、可交互运行的设计原则笔记。每个原则都遵循同一节奏：**一句话定义 → 生活类比 → 反例（违反）→ 正例（遵守）→ Python 实现技巧 → 动手练习**。

## 为什么要学设计原则？

写代码有两层目标：

- **让程序跑起来**（功能正确）——这是初学者的全部追求；
- **让程序"活得久"**（易于阅读、扩展、维护）——这是工程能力的分水岭。

需求一定会变。设计原则就是一组"抗压"指南：**当需求变化时，让你的改动尽量小、尽量局部、尽量不出错**。

## 六大原则一览

| 缩写 | 全称 | 中文 | 一句话 |
|---|---|---|---|
| **S**RP | Single Responsibility | 单一职责 | 一个类只有一个变化的原因 |
| **O**CP | Open/Closed | 开闭 | 对扩展开放，对修改关闭 |
| **L**SP | Liskov Substitution | 里氏替换 | 子类能无感替换父类 |
| **I**SP | Interface Segregation | 接口隔离 | 不被迫依赖不用的方法 |
| **D**IP | Dependency Inversion | 依赖倒置 | 依赖抽象，不依赖具体 |
| **LoD** | Law of Demeter | 迪米特法则（最少知识）| 只和直接朋友说话 |

前五个首字母拼成 **SOLID**；LoD（迪米特法则）常与它们并列，合称"六大设计原则"。

## 使用方式

- 从上到下依次运行代码单元，对照"反例 / 正例"体会差异；
- 遇到"练习"，先自己改一改，再看后面的参考写法；
- 多实验：把反例里的 `❌` 行解开注释、把正例里的抽象换成你自己的实现，观察行为。

## 0. 前置工具：耦合、内聚，以及 Python 的"接口"

在进入六大原则前，先建立两个度量衡和三件 Python 工具。后面所有原则本质都是在做同一件事：**提高内聚、降低耦合**。

### 两个度量衡

- **内聚（Cohesion）**：一个模块内部各元素"是否围绕同一件事"。内聚越高，模块越专注、越好懂、越好改。
- **耦合（Coupling）**：模块之间"相互依赖的强度"。耦合越低，改一个模块时牵连的模块越少。

> 经验法则：**追求高内聚、低耦合**。六大原则都是它的具体落地。

### Python 表达"接口/抽象"的三种方式

Python 没有像 Java 那样的 `interface` 关键字，但有三件等价工具：

In [ ]:
# 工具 1：鸭子类型（Duck Typing）
# "如果一个东西走起来像鸭子、叫起来像鸭子，那它就是鸭子。"
# 不声明类型，只要对象有需要的方法，就能用。

class Dog:
    def speak(self): return "汪汪"

class Cat:
    def speak(self): return "喵喵"

def make_sound(animal):       # 不检查类型，只关心有没有 speak()
    return animal.speak()

# Dog 和 Cat 没有继承任何共同的"动物基类"，但都有 speak()，所以都能被复用
print(make_sound(Dog()), make_sound(Cat()))

In [ ]:
# 工具 2：abc.ABC + @abstractmethod
# 显式定义"抽象基类 / 接口"，强制子类实现，且抽象类本身不能被实例化。

from abc import ABC, abstractmethod

class Animal(ABC):
    @abstractmethod
    def speak(self) -> str: ...

class Duck(Animal):
    def speak(self): return "嘎嘎"

# Animal()  # ❌ 取消注释会报错：TypeError: Can't instantiate abstract class
print(Duck().speak())

In [ ]:
# 工具 3：typing.Protocol（结构性接口）
# Protocol 是"鸭子类型的正式版"：定义一组方法签名，任何"长得像"的对象都算实现，
# 而且不要求显式继承。配合 @runtime_checkable 可以用 isinstance 检查。

from typing import Protocol, runtime_checkable

@runtime_checkable
class Speaker(Protocol):
    def speak(self) -> str: ...

class Robot:                    # 没有 (Speaker)，但"长得像"
    def speak(self): return "01001000"

print(isinstance(Robot(), Speaker))   # True

> **记一个选择建议**：需要"强制子类实现 + 拦截实例化"用 `ABC`；只想描述"某个形状的能力"、保持灵活用 `Protocol`。本笔记两种都会用到。

---

准备好了，下面正式进入 SOLID。

## 1. S — 单一职责原则（SRP）

> **一个类应该只有一个引起它变化的原因。**（A class should have one, and only one, reason to change.）

### 生活类比
一家餐厅里：**厨师**只管做菜，**服务员**只管端菜，**收银**只管结账。如果让服务员既端菜又做菜又结账，任何一个环节的变动（菜单更新、结账方式升级）都要改这一个人——出错概率陡增。

### 判断方法
问自己："**这个类会因为哪些不同的原因被修改？**" 如果答案超过一个，它就承担了过多职责。

### 反例：一个"什么都管"的员工类

In [ ]:
# ❌ 反例：Employee 同时承担了 4 个职责
class Employee:
    """变化原因 1：考勤规则变了 → 改这里
       变化原因 2：数据库换了 → 改这里
       变化原因 3：报表格式变了 → 改这里
       变化原因 4：加班费算法变了 → 改这里
    """
    def __init__(self, name, base_salary, hours):
        self.name = name
        self.base_salary = base_salary
        self.hours = hours

    def calculate_pay(self):                 # 职责 A：算工资
        overtime = max(0, self.hours - 40)
        return self.base_salary + overtime * 50

    def save_to_db(self, connection):        # 职责 B：持久化
        connection["employees"].append(
            {"name": self.name, "salary": self.base_salary}
        )

    def generate_report(self):               # 职责 C：生成报表（还顺手调了算工资）
        return f"员工 {self.name} 工时 {self.hours}，应发 {self.calculate_pay()}"

# 它能跑，但只要其中任何一个职责变动，整个类都要被打开、被回归测试。
db = {"employees": []}
emp = Employee("Alice", 8000, 48)
emp.save_to_db(db)
print(emp.generate_report())
print(db)

**问题**：算工资、存数据库、出报表，本该是三条互不相干的变更轴线，却被绑死在一个类里。任何一条轴线的改动都可能波及另外两条。

### 正例：按"变化的原因"拆分

In [ ]:
# ✅ 正例：每个类只有一个变化的原因
from dataclasses import dataclass

@dataclass
class EmployeeData:                # 职责：承载数据（数据结构变了才动它）
    name: str
    base_salary: float
    hours: int

class PayCalculator:               # 职责：工资规则（加班费算法变了只动这里）
    def calculate(self, emp: EmployeeData) -> float:
        overtime = max(0, emp.hours - 40)
        return emp.base_salary + overtime * 50

class EmployeeRepository:          # 职责：持久化（换数据库只动这里）
    def __init__(self):
        self.db = []
    def save(self, emp: EmployeeData):
        self.db.append({"name": emp.name, "salary": emp.base_salary})

class EmployeeReporter:            # 职责：呈现（报表格式变了只动这里）
    def __init__(self, calculator: PayCalculator):
        self.calculator = calculator
    def report(self, emp: EmployeeData) -> str:
        return f"员工 {emp.name} 工时 {emp.hours}，应发 {self.calculator.calculate(emp)}"

emp = EmployeeData("Alice", 8000, 48)
repo = EmployeeRepository(); repo.save(emp)
print(EmployeeReporter(PayCalculator()).report(emp))
print(repo.db)

### Python 小贴士
- `dataclasses.dataclass` 非常适合做"纯数据"类，天然符合 SRP——它只承载数据、不夹带行为。
- 拆分时常用**分层**思路：数据层（model）/ 业务层（service）/ 持久层（repository）/ 表现层（view 或 reporter）。
- 警惕"上帝类（God Object）"：字段特别多、方法横跨多个领域，往往就是 SRP 失守的信号。

### 📝 练习 1
1. 上面的 `EmployeeReporter` 通过构造函数接收了 `PayCalculator`，这其实已经悄悄用到了后面的 **DIP**。思考：如果把 `PayCalculator()` 直接 `new` 在 reporter 内部，会有什么坏处？
2. 写一个违反 SRP 的 `User` 类（同时管登录、管发邮件、管存数据库），再把它拆成 3 个职责单一的类。

## 2. O — 开闭原则（OCP）

> **软件实体应该对扩展开放，对修改关闭。**（Open for extension, closed for modification.）

### 生活类比
电脑的 USB 接口：你想接鼠标、U 盘、键盘，**不需要拆开电脑改电路**——只要新设备符合 USB 标准，插上就能用。接口"对修改关闭"，但"对扩展开放"。

### 核心思路
**用抽象（抽象类 / 接口 / Protocol）和多态承接变化**。新增一种行为 = 新增一个新类，而不是去改老代码里那串 `if/elif`。

### 反例：类型分支写死在方法里

In [ ]:
# ❌ 反例：每加一种折扣，都要打开这个方法改 if/elif
class DiscountCalculator:
    def calculate(self, price, discount_type):
        if discount_type == "normal":
            return price
        elif discount_type == "vip":
            return price * 0.8
        elif discount_type == "svip":
            return price * 0.6
        # 想加"新人价"？只能改这里 ❌ —— 修改了已通过测试的老代码
        else:
            raise ValueError(f"未知折扣类型: {discount_type}")

calc = DiscountCalculator()
print(calc.calculate(100, "normal"), calc.calculate(100, "vip"), calc.calculate(100, "svip"))

**问题**：折扣规则会不断增长。每加一条规则就要改 `calculate`，可能误伤已有分支；分支越多越像乱麻。

### 正例：把"折扣"抽象成可替换的策略

In [ ]:
# ✅ 正例：折扣 = 可替换的策略；新增折扣 = 新增一个类，旧代码一行不改
from abc import ABC, abstractmethod

class Discount(ABC):                     # 抽象：定义"折扣"这个能力的接口
    @abstractmethod
    def apply(self, price: float) -> float: ...

class NormalDiscount(Discount):
    def apply(self, price): return price
class VipDiscount(Discount):
    def apply(self, price): return price * 0.8
class SvipDiscount(Discount):
    def apply(self, price): return price * 0.6

class NewcomerDiscount(Discount):        # ✨ 新增策略：完全不碰上面的类
    def apply(self, price): return price * 0.5

# 用"注册表（字典）"代替 if/elif：新增类型 = 加一行映射
DISCOUNTS = {
    "normal": NormalDiscount(),
    "vip": VipDiscount(),
    "svip": SvipDiscount(),
    "newcomer": NewcomerDiscount(),
}

def calculate(price, discount_type, registry=DISCOUNTS):
    return registry[discount_type].apply(price)

print(calculate(100, "normal"), calculate(100, "vip"), calculate(100, "newcomer"))

### Python 小贴士
- 常见落地手段：**策略模式**（如上）、**模板方法**、**注册表/插件字典**。
- Python 还可以更轻：用"函数即策略"，`DISCOUNTS = {"vip": lambda p: p*0.8, ...}`。复杂行为才升级成类。
- 判断信号：**同一段 `if/elif/elif` 里每个分支都在做"同一件事的不同版本"** → 强烈的 OCP 重构信号。

### 📝 练习 2
1. 把上面的折扣系统从"基于类"改成"基于函数 + 字典"，体会两种写法的取舍。
2. 想象一个物流系统，运费按"陆运/海运/空运"用 `if/elif` 计算。用 OCP 重构它，使得新增"高铁运输"无须改老代码。

## 3. L — 里氏替换原则（LSP）

> **所有引用父类的地方，必须能透明地使用子类对象，而程序行为不出错。**（子类必须能替换其父类。）

### 生活类比
你订了一个"杯子"，商家发来"玻璃杯""陶瓷杯""保温杯"你都能接受——因为它们都满足"杯子"的契约（能装液体、能从杯口喝）。但如果商家发来一个"漏底的杯子"，它**虽然叫杯子，却不能用**，就破坏了替换性。

### 行为契约（关键）
"继承"不只是"is-a 关系"，更是**行为契约的承诺**。子类重写方法时：
- **前置条件**（对参数的要求）不能比父类更严；
- **后置条件**（对返回值/状态的承诺）不能比父类更弱；
- **不能抛出父类不会抛的新异常**。

换句话说：**使用者不该因为换成子类而"被惊吓"。**

### 经典反例：正方形继承长方形

In [ ]:
# ❌ 反例：正方形"是"长方形，但行为不能替换父类
class Rectangle:
    def __init__(self, width, height):
        self.width, self.height = width, height
    def set_width(self, w):  self.width = w
    def set_height(self, h): self.height = h
    def area(self):          return self.width * self.height

class Square(Rectangle):
    """为了保持"四边相等"，重写 setter 时偷偷改了另一条边 → 破坏了父类契约"""
    def set_width(self, w):
        self.width = w;  self.height = w
    def set_height(self, h):
        self.width = h;  self.height = h

# 一段"面向父类 Rectangle 编程"的代码，假设宽高可独立设置
def resize(r: Rectangle, w, h):
    r.set_width(w); r.set_height(h)
    assert r.area() == w * h, f"期望面积 {w*h}，实际 {r.area()}"

resize(Rectangle(2, 3), 4, 5)
print("Rectangle 通过 ✅")

try:
    resize(Square(2, 2), 4, 5)            # 用子类替换父类
except AssertionError as e:
    print("Square 替换失败 ❌：", e)
# 教训：Square 不能安全替换 Rectangle → 这个继承关系本身是错的。

**问题**：从"分类学"看正方形确实是长方形，但从**行为契约**看，`Square` 的 setter 有"连带副作用"，违反了父类"宽高独立"的隐含承诺。继承要忠于**行为**而非**直觉上的分类**。

### 正例：让两者共享更恰当的抽象

In [ ]:
# ✅ 正例：不要强行继承。提取一个只承诺 area() 的抽象 Shape，让两者平级
from abc import ABC, abstractmethod

class Shape(ABC):
    """只承诺'能算面积'，不承诺可独立设置宽高"""
    @abstractmethod
    def area(self) -> float: ...

class Rectangle(Shape):
    def __init__(self, width, height):
        self.width, self.height = width, height
    def set_width(self, w):  self.width = w
    def set_height(self, h): self.height = h
    def area(self):          return self.width * self.height

class Square(Shape):
    def __init__(self, side):
        self.side = side
    def set_side(self, s):   self.side = s
    def area(self):          return self.side ** 2

# 现在面向 Shape 编程，Rectangle 和 Square 都能安全替换
shapes = [Rectangle(4, 5), Square(3)]
print([s.area() for s in shapes])

### Python 小贴士
- 另一个高频陷阱：**`class Stack(list)`**。栈应该是"后进先出"，继承 `list` 后却暴露了 `insert`、按索引访问等能力，使用者可以用子类做出违反栈语义的操作 → 同样违反 LSP。栈更应**组合**一个 list，而非继承它。
- 经验：当你为了让子类"成立"而**不断给父类方法加注释/限制**时，多半是 LSP 在报警。
- 子类可以**加强**行为，但不能**削弱**父类已承诺的行为。

### 📝 练习 3
1. 用 `assert` 写一个测试，证明上面的 `Square(Shape)` 能和 `Rectangle(Shape)` 一样通过"算面积"的使用场景。
2. 设计一个反例：`Bird` 有 `fly()`，`Penguin(Bird)` 不会飞。说明这为何违反 LSP，并给出修正方案（提示：重新划分抽象层级，把"飞"单独抽离）。

## 4. I — 接口隔离原则（ISP）

> **客户端不应该被迫依赖它不使用的方法。**（多个专用的接口，好过一个"胖接口"。）

### 生活类比
一个"瑞士军刀"式的复印机接口：打印、扫描、传真、装订全塞在一起。你买了一台只能打印的小打印机，说明书却要求你按"扫描""传真"按钮——你只能贴个纸条写"本机无此功能"。把这些功能**拆成独立的小接口**，各取所需，互不勉强。

### 反例：一个臃肿的胖接口

In [ ]:
# ❌ 反例：胖接口 Machine 把打印/扫描/传真全捆在一起
from abc import ABC, abstractmethod

class Machine(ABC):
    @abstractmethod
    def print(self, doc): ...
    @abstractmethod
    def scan(self, doc): ...
    @abstractmethod
    def fax(self, doc): ...

class SimplePrinter(Machine):
    """只能打印的打印机，却被逼着实现 scan/fax"""
    def print(self, doc): print(f"打印: {doc}")
    def scan(self, doc):  raise NotImplementedError("这台机器不能扫描")  # ❌ 被迫依赖
    def fax(self, doc):   raise NotImplementedError("这台机器不能传真")

p = SimplePrinter()
p.print("report.pdf")
# p.scan(...) 会在运行时才崩 —— 客户端被迫依赖它根本用不到的方法

**问题**：`SimplePrinter` 的使用者明明只调用 `print`，却因为接口胖，被迫"拥有" scan/fax 的方法（哪怕只是抛异常）。一旦 `Machine` 接口加一个 `staple`（装订），所有实现类全要跟着改。

### 正例：按"真正需要的能力"拆分接口

In [ ]:
# ✅ 正例：把能力拆成独立的小接口（Python 里用 Protocol 天然合适）
from typing import Protocol

class Printer(Protocol):
    def print(self, doc) -> None: ...
class Scanner(Protocol):
    def scan(self, doc) -> None: ...
class FaxMachine(Protocol):
    def fax(self, doc) -> None: ...

class SimplePrinter:                 # 只实现需要的，干净利落
    def print(self, doc): print(f"打印: {doc}")

class MultiFunctionPrinter:          # 多功能机实现全部接口
    def print(self, doc): print(f"打印: {doc}")
    def scan(self, doc):  print(f"扫描: {doc}")
    def fax(self, doc):   print(f"传真: {doc}")

# 函数只声明自己需要的那个窄接口 → SimplePrinter / MFP 都能传入
def run_print(p: Printer, doc):
    p.print(doc)

run_print(SimplePrinter(), "report.pdf")
run_print(MultiFunctionPrinter(), "invoice.pdf")

### Python 小贴士
- **Protocol 是 ISP 的天然载体**：你只为每个能力定义一个小 Protocol，类型注解里写哪个，就只依赖哪个。
- ISP 与 SRP 是一对"镜像"：SRP 讲"一个类别干太多"，ISP 讲"一个接口别塞太多"。
- 信号：实现类里出现一堆 `raise NotImplementedError` → 接口该拆了。

### 📝 练习 4
1. 上面的 `MultiFunctionPrinter` 同时满足三个 Protocol。写代码用 `isinstance`（配合 `@runtime_checkable`）验证它既是 `Printer` 又是 `Scanner`。
2. 设计一个胖接口 `Worker`（含 `work`、`eat`、`sleep`），再拆成能区分"人类工人"和"机器人工人"的小接口（机器人不吃饭不睡觉）。

## 5. D — 依赖倒置原则（DIP）

> 1. **高层模块不应依赖低层模块，二者都应依赖抽象。**
> 2. **抽象不应依赖细节，细节应依赖抽象。**

### 生活类比
电脑主板上有"USB 标准"这个抽象。主板（高层）不认识具体哪个鼠标/键盘（低层），鼠标/键盘也都按 USB 标准来造。**谁也不依赖谁的具体实现，大家都依赖"USB 标准"这个抽象**——于是任意鼠标都能插任意主板。

### 关键手段：依赖注入（Dependency Injection）
"依赖抽象"的落地方式，是不在类内部 `new` 具体依赖，而是**从外部把依赖传进来**（构造函数参数最常见）。这样高层就只认识抽象，不认识具体。

### 反例：高层直接依赖低层具体类

In [ ]:
# ❌ 反例：NotificationService 直接 new 了一个 EmailSender
class EmailSender:
    def send(self, message): print(f"[邮件] {message}")

class NotificationService:
    def __init__(self):
        self.sender = EmailSender()     # ❌ 高层写死了对低层具体类的依赖
    def notify(self, message):
        self.sender.send(message)

NotificationService().notify("订单已发货")

# 痛点：① 想改成短信通知？必须修改 NotificationService 内部；
#       ② 想测试？没法塞一个"假的 sender"进去，必须真的发邮件。

**问题**：高层 `NotificationService` 和低层 `EmailSender` 强耦合。低层一变（换实现、换渠道），高层跟着遭殃；高层也变得难以单元测试。

### 正例：双方都依赖抽象，具体实现从外部注入

In [ ]:
# ✅ 正例：抽象出 MessageSender，高层和低层都依赖它
from typing import Protocol

class MessageSender(Protocol):            # 抽象：发送能力
    def send(self, message: str) -> None: ...

class EmailSender:                        # 低层细节：依赖抽象（实现 Protocol）
    def send(self, message): print(f"[邮件] {message}")

class SmsSender:                          # 新的低层实现：无须改高层
    def send(self, message): print(f"[短信] {message}")

class NotificationService:
    def __init__(self, sender: MessageSender):   # 依赖抽象，不依赖具体；依赖从外部注入
        self.sender = sender
    def notify(self, message):
        self.sender.send(message)

# 高层不再关心用邮件还是短信——由调用方决定（注入）
NotificationService(EmailSender()).notify("订单已发货")
NotificationService(SmsSender()).notify("验证码 8888")

# 收益：① 扩展（加新发送方式）不修改高层；② 测试时可注入"假 sender"，无需联网
class FakeSender:
    def __init__(self): self.sent = []
    def send(self, message): self.sent.append(message)

fake = FakeSender()
NotificationService(fake).notify("这是一条测试消息")
print("测试捕获到：", fake.sent)

### Python 小贴士
- 依赖注入最简形式就是**构造函数传参**，不需要任何框架。复杂工程才上 `dependency-injector` 之类的容器。
- DIP 常和 OCP 联手：因为高层依赖抽象，所以新增"另一种实现"属于扩展（OCP 满足），且高层无须改（DIP 满足）。
- 信号：类里出现 `self.x = SomeConcreteClass()` 且这个 `SomeConcreteClass` 还可能被替换 → 抽出去、注入进来。

### 📝 练习 5
1. 给 `NotificationService` 再加一种 `WechatSender`，验证高层代码一行都不用改。
2. 把练习 1 里 `EmployeeReporter` 直接 `new PayCalculator()` 的版本，改造成依赖注入的版本，并体会"可测试性"的提升。

## 6. LoD — 迪米特法则 / 最少知识原则（Law of Demeter）

> **一个对象应该对其它对象保持最少的了解。** 通俗版：**只和你的"直接朋友"说话，别跟陌生人搭话。**

### 生活类比
你在餐馆结账，你只会对顾客说"请付 50 元"，而**不会**直接伸手去掏顾客的钱包。顾客怎么付（现金？刷卡？掏钱包？）是顾客自己的事——你只发出"付款"这个**高层请求**，不操心对方的内部结构。

### "直接朋友"的范围
某方法 `M` 内部，只允许调用以下对象的方法：
1. `self`（对象自身）；
2. `M` 的**参数**；
3. `M` 内部**创建**的对象；
4. `self` 的**直接字段**（组件）。

除此之外的对象，都是"**陌生人**"——不该去碰。

### 反例：火车链（train wreck）

In [ ]:
# ❌ 反例：收银员直接伸手掏顾客的钱包——跨越了两层"陌生人"
class Wallet:
    def __init__(self, balance): self.balance = balance
    def withdraw(self, amount):
        self.balance -= amount
        return amount

class Customer:
    def __init__(self, name, wallet):
        self.name = name
        self.wallet = wallet          # ❌ 内部细节（钱包）被公开暴露

class Cashier:
    def checkout(self, customer, amount):
        # 火车链：customer.wallet.withdraw(...)
        # 收银员不仅知道顾客有钱包，还知道钱包有 withdraw(amount) —— 知道得太多
        customer.wallet.withdraw(amount)
        print(f"向 {customer.name} 收款 {amount}")

Cashier().checkout(Customer("Alice", Wallet(1000)), 200)

# 风险：一旦 Customer 把 wallet 改名、或换成 BankCard，Cashier 全线崩溃。

**问题**：`Cashier` 耦合了 `Customer` 的内部结构（钱包），又耦合了 `Wallet` 的方法。链子越长，牵连面越大——`a.getB().getC().do()` 这种"火车链"是经典坏味道。

### 正例：只和直接朋友说话（Tell, Don't Ask）

In [ ]:
# ✅ 正例：让顾客自己付款，收银员只和"顾客"这一个直接朋友对话
class Wallet:
    def __init__(self, balance): self.balance = balance
    def withdraw(self, amount):
        self.balance -= amount
        return amount

class Customer:
    def __init__(self, name, wallet):
        self.name = name
        self._wallet = wallet         # 私有化，不再对外暴露
    def pay(self, amount):            # 暴露一个高层行为，内部细节自己处理
        return self._wallet.withdraw(amount)

class Cashier:
    def checkout(self, customer: Customer, amount):
        customer.pay(amount)          # 只和直接朋友 customer 对话
        print(f"向 {customer.name} 收款 {amount}")

Cashier().checkout(Customer("Bob", Wallet(1000)), 200)
# 现在 Customer 内部把钱包换成银行卡、还是加优惠券，Cashier 都无感。

### Python 小贴士
- 等价口诀：**"Tell, Don't Ask"（告诉它做，而不是问出来再做）**。与其把数据问出来在本地判断/操作，不如请对象自己完成动作。
- 注意：DTO / 数据类（`dataclass` 纯数据）天然暴露字段，**不算**违反 LoD——LoD 针对"有行为的对象"。
- 链式调用 `a.b().c()` 要分清：如果是**流畅接口**（如 `df.filter().sort()`，每一步都返回设计好的同体系对象）则没问题；如果是**深挖内部结构**（如 `order.get_customer().get_wallet().withdraw()`）就是坏味道。

### 📝 练习 6
1. 找出并改写这行违反 LoD 的代码：`tax = employee.get_department().get_manager().get_salary() * 0.1`。提示：给一个合适的对象增加一个高层方法（如 `department.manager_tax()`）。
2. 思考：`print(person.address.city.name)` 是否违反 LoD？什么情况下可以接受？

## 7. 综合实战：给一段"坏味道"代码做体检

下面这段代码"能跑"，但几乎每行都在违反某条原则。**先运行它**，然后对照注释，自己试着把每条坏味道对应到具体原则（答案在下一个单元）。

In [ ]:
# ❌ 综合反例：一个"上帝类"，集中暴露多原则问题
class OrderProcessor:
    def __init__(self, order):
        self.order = order
    def process(self, payment_type):
        if not self.order["items"]:                              # 🟥 坏味道 A
            raise ValueError("订单为空")
        total = sum(i["price"] * i["qty"] for i in self.order["items"])

        if payment_type == "alipay":                             # 🟥 坏味道 B
            print(f"支付宝支付 {total}")
        elif payment_type == "wechat":
            print(f"微信支付 {total}")
        else:
            raise ValueError("不支持的支付方式")

        email = self.order["customer"]["email"]                 # 🟥 坏味道 C
        print(f"[邮件 -> {email}] 订单已确认")                   # 🟥 坏味道 D

order = {"customer": {"name": "Alice", "email": "alice@x.com"},
         "items": [{"name": "书", "price": 50, "qty": 2}]}
OrderProcessor(order).process("alipay")

### 体检报告（对答案）

| 坏味道 | 违反的原则 | 理由 |
|---|---|---|
| A：校验"订单是否为空"混在编排流程里 | **SRP** | 编排类不应承担业务校验职责 |
| B：`if/elif` 区分支付方式 | **OCP** | 新增支付方式必须改这里；应抽象成可替换策略 |
| C：`order["customer"]["email"]` 深挖内部结构 | **LoD** | 跨层钻进内部数据，耦合了订单与顾客的字段布局 |
| D：通知方式写死成邮件 `print` | **DIP** | 高层写死了通知渠道；应依赖抽象并注入 |

> 顺带：把数据塞在裸 `dict` 里到处传，也削弱了类型安全——下面的正例用 `dataclass` 建模，可读性立刻提升。

### 重构后：各原则各司其职

In [ ]:
# ✅ 综合正例：用前面学过的全部原则重构
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Protocol

# --- 数据模型（SRP：只承载数据）---
@dataclass
class Customer:
    name: str
    email: str

@dataclass
class Order:
    customer: Customer
    items: list                       # [{"name", "price", "qty"}]
    def total(self) -> float:         # LoD：把"算总价"封装成自己的高层方法
        return sum(i["price"] * i["qty"] for i in self.items)

# --- 支付抽象（OCP + DIP）：新增支付 = 新增类 ---
class Payment(ABC):
    @abstractmethod
    def pay(self, amount: float) -> None: ...
class Alipay(Payment):
    def pay(self, amount): print(f"支付宝支付 {amount}")
class WechatPay(Payment):
    def pay(self, amount): print(f"微信支付 {amount}")

# --- 通知抽象（DIP）：高层依赖抽象，不依赖具体 ---
class Notifier(Protocol):
    def notify(self, customer: Customer, message: str) -> None: ...
class EmailNotifier:
    def notify(self, customer, message):
        print(f"[邮件 -> {customer.email}] {message}")

# --- 高层编排（SRP：只做流程编排，其它都委托出去）---
class OrderProcessor:
    def __init__(self, payment: Payment, notifier: Notifier):   # DIP：依赖注入
        self.payment = payment
        self.notifier = notifier
    def process(self, order: Order):
        if not order.items:                                       # 校验也由高层统一把关
            raise ValueError("订单为空")
        amount = order.total()                                    # LoD：只问直接朋友
        self.payment.pay(amount)                                  # OCP/DIP：交给抽象
        self.notifier.notify(order.customer, "订单已确认，请等待发货")

# 组装：高层不关心任何具体细节，只把"积木"插起来
order = Order(Customer("Alice", "alice@x.com"),
              [{"name": "书", "price": 50, "qty": 2}])
OrderProcessor(Alipay(), EmailNotifier()).process(order)

# 想换微信支付 + 加一个测试用假通知？只换注入的对象，OrderProcessor 一行不改：
class FakeNotifier:
    def __init__(self): self.log = []
    def notify(self, customer, message): self.log.append((customer.name, message))

fake = FakeNotifier()
OrderProcessor(WechatPay(), fake).process(order)
print("测试捕获：", fake.log)

### 原则之间的联系

六大原则不是孤岛，它们彼此补强：

- **SRP** 是地基：职责清晰了，其它原则才好谈。
- **OCP** 与 **DIP** 常成对出现：依赖抽象（DIP）使得"新增实现 = 扩展"成为可能（OCP）。
- **LSP** 是多态安全的保证：没有它，OCP 那套"面向抽象编程"会在运行时翻车。
- **ISP** 让"抽象"保持精简：接口越小，DIP 里"注入正确的实现"就越容易。
- **LoD** 是耦合度的最后一道防线：即便抽象设计得再好，深挖内部结构也会把耦合悄悄加回来。

> 经验：**先把职责分清（SRP/ISP），再用抽象承接变化（OCP/DIP/LSP），最后用最少知识收紧耦合（LoD）**。

---

## 速查表

| 原则 | 一句话 | 关键词 | 常见代码味道 |
|---|---|---|---|
| **SRP** 单一职责 | 一个类只有一个变化的原因 | 职责、内聚 | 上帝类、方法过长、多领域混居 |
| **OCP** 开闭 | 对扩展开放，对修改关闭 | 抽象、多态、策略 | `if/elif` 按类型分支 |
| **LSP** 里氏替换 | 子类能无感替换父类 | 行为契约、一致性 | 子类重写破坏父类语义、`NotImplementedError` |
| **ISP** 接口隔离 | 不被迫依赖不用的方法 | 窄接口、能力拆分 | 胖接口、被迫实现空方法 |
| **DIP** 依赖倒置 | 依赖抽象，不依赖具体 | 抽象、依赖注入 | 类内部 `new` 具体依赖、难单测 |
| **LoD** 迪米特 | 只和直接朋友说话 | 最少知识、封装 | 火车链 `a.b().c().d()` |

## 一句话记忆

> **S**plit 职责（SRP）→ **O**pen to extend（OCP）→ **L**eave substitutable（LSP）→ **I**solate 接口（ISP）→ **D**epend on 抽象（DIP）→ **D**emeter 最少知识（LoD）。

## 延伸阅读

- Robert C. Martin《Agile Software Development, Principles, Patterns, and Practices》（SOLID 的提出者）
- 《Refactoring》Fowler —— 识别与消除"坏味道"
- Python 文档：[`abc`](https://docs.python.org/3/library/abc.html)、[`typing.Protocol`](https://docs.python.org/3/library/typing.html#typing.Protocol)、[`dataclasses`](https://docs.python.org/3/library/dataclasses.html)

---

*学完六大原则，下一步可以进入"设计模式"——它们正是这些原则在具体场景下的经典组合。*